# Day 1: Knowledge Graph Exploration
Verify graph stats, visualise a sample patient subgraph, and manually test 3 benchmark questions.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

from graph_retriever import load_graph
from collections import Counter

G = load_graph()
print(f'Total nodes : {G.number_of_nodes():,}')
print(f'Total edges : {G.number_of_edges():,}')

node_types = Counter(d['node_type'] for _, d in G.nodes(data=True))
edge_types = Counter(d.get('edge_type', '?') for _, _, d in G.edges(data=True))

print('\nNodes by type:')
for t, c in sorted(node_types.items()):
    print(f'  {t:<14}: {c:>6,}')

print('\nEdges by type:')
for t, c in sorted(edge_types.items()):
    print(f'  {t:<16}: {c:>6,}')

## Sample patient subgraph visualisation (pyvis)

In [ ]:
from pyvis.network import Network
from graph_retriever import resolve_patient_node
import networkx as nx

SAMPLE_PATIENT = '9358d6df'   # first benchmark patient
SAMPLE_WAVE    = 5

patient_node = resolve_patient_node(G, SAMPLE_PATIENT)
print(f'Patient node: {patient_node}')

# Build a small subgraph: patient + wave node + all nodes in that wave
sub_nodes = {patient_node, f'wave_{SAMPLE_WAVE}'}
for neighbor in G.successors(patient_node):
    nd = G.nodes[neighbor]
    if nd.get('wave') == SAMPLE_WAVE:
        sub_nodes.add(neighbor)

SG = G.subgraph(sub_nodes).copy()
print(f'Subgraph: {SG.number_of_nodes()} nodes, {SG.number_of_edges()} edges')

# Colour map
COLOURS = {
    'Patient':     '#4e79a7',
    'Wave':        '#f28e2b',
    'Condition':   '#e15759',
    'Observation': '#76b7b2',
    'Medication':  '#59a14f',
}

net = Network(height='600px', width='100%', notebook=True, directed=True)
net.barnes_hut()

for node_id, nd in SG.nodes(data=True):
    ntype = nd.get('node_type', 'Unknown')
    label = nd.get('description', nd.get('short_id', node_id))[:40]
    net.add_node(node_id, label=label, color=COLOURS.get(ntype, '#aaa'),
                 title=f"{ntype}\n{node_id}")

for src, dst, ed in SG.edges(data=True):
    net.add_edge(src, dst, label=ed.get('edge_type', ''))

out_path = '../data/sample_subgraph.html'
net.write_html(out_path)
print(f'Saved to {out_path}')

# Display inline
from IPython.display import IFrame
IFrame(out_path, width='100%', height=620)

## Manual verification – 3 benchmark questions

In [ ]:
from graph_retriever import format_subgraph_context
import pandas as pd

bm = pd.read_csv('../benchmark/qa_pairs_final.csv')

TEST_IDS = ['LOOKUP_001', 'LOOKUP_002', 'TREND_001']

def parse_waves(wave_ref):
    from graph_rag_pipeline import parse_wave_reference
    return parse_wave_reference(str(wave_ref))

for qid in TEST_IDS:
    row = bm[bm['question_id'] == qid].iloc[0]
    pid   = row['patient_id']
    waves = parse_waves(row['wave_reference'])
    mode  = 'comparison' if len(waves) >= 2 else ('full_history' if not waves else 'specific')
    ctx   = format_subgraph_context(G, pid, waves or None, mode)
    print('=' * 60)
    print(f'Q [{qid}]: {row["question"]}')
    print(f'Gold    : {row["answer"]}')
    print('--- Context ---')
    print(ctx[:1000])
    print()

## Run a single question through the full pipeline

In [ ]:
from graph_rag_pipeline import run_pipeline

row = bm[bm['question_id'] == 'LOOKUP_001'].iloc[0]
result = run_pipeline(
    question=row['question'],
    patient_id=row['patient_id'],
    wave_reference=row['wave_reference'],
    category=row['category'],
)
print(f"Q    : {row['question']}")
print(f"Gold : {row['answer']}")
print(f"Pred : {result['answer']}")